# 🎨 NOVA v2.7.0 — Entrenamiento CREATIVO AVANZADO (V2 - Fixed)
**Objetivo:** Máxima fluidez en Poesía, Novela, Chistes e Ideas de Negocio.

**Nota:** Se ha corregido el error de 'StopIteration' asegurando que el dataset nunca esté vacío.

**Instrucciones:** Ejecutar todo. Tarda ~25-30 minutos.

In [ ]:
print("⏳ Instalando Unsloth y herramientas de entrenamiento...")
import os
os.system("pip install unsloth")
os.system("pip install --no-deps trl peft accelerate bitsandbytes datasets")

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. CARGA DE MODELO
print("⏳ Cargando cerebro base (DeepSeek 1.3B)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="deepseek-ai/deepseek-coder-1.3b-instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=32, target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32, lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth", random_state=3407,
)

# 2. DATASET CREATIVO EXPANDIDO
print("⏳ Cargando base de datos 'SomosNLP'...")
dataset_full = load_dataset("somosnlp/somos-clean-alpaca-es", split="train")

# DEBUG: Ver estructura real del dataset
print(f"Estructura del ejemplo: {dataset_full[0].keys()}")

keywords = [
    "poema", "poesia", "verso", "rima", "oda", "soneto",
    "novela", "capitulo", "narrativa", "cuento", "historia", "relato",
    "guion", "teatro", "personaje", "trama", 
    "chiste", "humor", "broma", "gracioso", "comedia",
    "startup", "negocio", "empresa", "idea", "emprendimiento", "estrategia",
    "inventa", "imagina", "crea", "brainstorming", "escribe"
]

def is_super_creative(example):
    # Intentar obtener contenido de varios campos comunes
    instr = str(example.get("instruction", example.get("prompt", "")))
    out = str(example.get("output", example.get("response", "")))
    text = (instr + " " + out).lower()
    return any(k in text for k in keywords)

print("⏳ Filtrando ejemplos creativos...")
creative_ds = dataset_full.filter(is_super_creative)

# FALLBACK: Si por alguna razón el filtro falla (ej. tildes), tomar una muestra aleatoria
if len(creative_ds) == 0:
    print("⚠️ Advertencia: Filtro devolvió 0. Usando muestra general para evitar error.")
    creative_ds = dataset_full.shuffle(seed=3407).select(range(2000))
else:
    max_examples = min(10000, len(creative_ds))
    creative_ds = creative_ds.select(range(max_examples))

print(f"✅ Procesando {len(creative_ds)} ejemplos para entrenamiento.")

def format_prompt(e):
    instr = e.get("instruction", e.get("prompt", ""))
    inp = e.get("input", "")
    out = e.get("output", e.get("response", ""))
    return {"text": f"### Instrucción:\n{instr}\n\n### Entrada:\n{inp}\n\n### Respuesta:\n{out}{tokenizer.eos_token}"}

dataset = creative_ds.map(format_prompt)

# 3. ENTRENAMIENTO
print(f"🚀 INICIANDO TURBO-TUNING ({len(dataset)} ejemplos)...")
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset, dataset_text_field="text",
    max_seq_length=2048, dataset_num_proc=2, packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4, warmup_steps=10,
        num_train_epochs=1, learning_rate=1e-4, fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(), logging_steps=10, optim="adamw_8bit",
        weight_decay=0.01, lr_scheduler_type="linear", seed=3407, output_dir="outputs_creative_fixed",
    ),
)
trainer.train()

# 4. EXPORTAR GGUF
print("✅ Proceso completo. Generando archivo para Ollama...")
model.save_pretrained_gguf("nova-ultimate-creative", tokenizer, quantization_method="q4_k_m")

from google.colab import files
try:
    files.download("nova-ultimate-creative/unsloth.Q4_K_M.gguf")
except:
    print("⚠️ Descarga manual requerida en la carpeta 'nova-ultimate-creative'.")